# Eastport, Maine (nearshore) — site pipeline

**Config:** `data/eastport/config.yaml`

**Technologies:** wind, solar, hydro (tidal / ORPC TidGen)

Fishing-industry constraints — verify tidal/hydro CF is plausible, not wave.

Run cells **top to bottom**. Each markdown cell explains the **next code cell** — what it does, what to inspect in the output, and what counts as a pass.

Track overall audit progress in Obsidian (`FlexiMORP Calculation Audit.md`). These notebooks are the lab workbook, not the checklist.

**Important:** Do not catch errors and substitute dummy numbers (unlike `notebooks/alaska_analysis.ipynb`).

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("error", category=RuntimeWarning)


def _repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "fleximorpv2").is_dir():
            return candidate
    raise RuntimeError(
        "Could not find fleximorp-project root. "
        "Open Jupyter from the repo or a notebooks/ subdirectory."
    )


_repo = _repo_root()
_audit = _repo / "notebooks" / "audit"
for path in (_repo, _audit):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from _audit_helpers import (
    REPO_ROOT,
    OUTPUT_DIR,
    SITE_OUTPUT_DIR,
    reload_fleximorp,
    assert_close,
    assert_energy_balance,
    assert_cf_bounds,
    assert_positive,
)

reload_fleximorp()
np.random.seed(42)
print(f"Repo root: {REPO_ROOT}")
print(f"Audit outputs: {OUTPUT_DIR}")
print(f"Site outputs: {SITE_OUTPUT_DIR}")

## Step 1 — Load and validate config

**Run the next cell** to load `data/eastport/config.yaml`.

**Pass if:** `validate_config()` succeeds and enabled techs are `wind, solar, hydro (tidal / ORPC TidGen)`.

In [ ]:
from fleximorpv2.config import load_config, validate_config

config = load_config("eastport")
validate_config(config)
print(f"Site: {config.name}")
print(f"Coords: {config.coordinates}")
print(f"Techs: {config.get_enabled_technologies()}")
print(f"Discount rate: {config.economic.get('discount_rate')}")
print(f"Max capacity (config): {config.optimization.get('constraints', {}).get('max_total_capacity', 'not set')}")

## Step 2 — Inspect synthetic resource data

**Run the next cell.** Wave is disabled in config; hydro represents tidal stream.

**Pass if:** means are finite and physically plausible for the site (print values and judge — no API keys required).

In [ ]:
from fleximorpv2.utils.data_loader import APIDataLoader

loader = APIDataLoader(config)
resource = loader.load_weather_data(
    coordinates=config.coordinates,
    technologies=config.get_enabled_technologies(),
)
print("Wind mean (m/s):", float(resource.wind_speed.mean()))
print("Solar mean (W/m²):", float(resource.solar_irradiance.mean()))
if len(resource.wave_height):
    print("Wave Hs mean (m):", float(resource.wave_height.mean()))
print("Temperature mean (°C):", float(resource.temperature.mean()))

## Step 3 — Baseline optimization

**Run the next cell.**

Compare `target_value=200_000` to printed `annual_energy` — confirm whether the engine treats both as kWh or MWh (see notebook 08).

**Pass if:** optimisation completes, LCOE > 0, and capacity respects config limits.

In [ ]:
from fleximorpv2.baseline_optimization import BaselineOptimization

config.uncertainty["monte_carlo_runs"] = 10
results = BaselineOptimization(config).optimize("production", 200_000, method="scipy")

assert_positive(results.financial_metrics["lcoe"], label="LCOE")
print("Target:", 200_000, "| Annual energy:", results.technical_metrics["annual_energy"])
print("Financial:", results.financial_metrics)
print("Technical:", results.technical_metrics)
print("Capacities:", results.technology_capacities)
optimal_design = results.optimal_design

## Step 4 — Uncertainty (Monte Carlo)

**Run the next cell** with `n_runs=50` for a quick pass; re-run locally with 500 for convergence.

**Pass if:** mean LCOE is same order of magnitude as baseline LCOE.

In [ ]:
from fleximorpv2.uncertainty_analysis import UncertaintyAnalysis

analyzer = UncertaintyAnalysis(config)
uncertainty = analyzer.analyze_uncertainty(
    baseline_design=optimal_design,
    sampling_method="monte_carlo",
    reoptimize=False,
)
print("Baseline LCOE:", results.financial_metrics["lcoe"])
print("Mean LCOE under uncertainty:", uncertainty.mean_performance.get("lcoe"))
print("Risk metrics:", uncertainty.risk_metrics)

## Step 5 — Optional extensions

Run only after Steps 1–4 pass:
- `fleximorpv2/flexible_design.py` — real options value should be ≥ 0
- `fleximorpv2/sensitivity_analysis.py` — parameter rankings stable across two runs
- Save plots to `notebooks/sites/outputs/`

In [ ]:
# Optional — flexible design and sensitivity
